In [ ]:
from collections import defaultdict, Counter
import math
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
train_sentences = [
    [("Mary", "NNP"), ("Jane", "NNP"), ("can", "MD"), ("see", "VB"), ("will", "MD")],
    [("Spot", "NNP"), ("will", "MD"), ("see", "VB"), ("Mary", "NNP")],
    [("Will", "NNP"), ("Jane", "NNP"), ("spot", "VB"), ("Mary", "NNP")],
    [("Mary", "NNP"), ("will", "MD"), ("pat", "VB"), ("spot", "NNP")],
]

test_sentences = [
    [("Will", "NNP"), ("can", "MD"), ("pot", "VB"), ("Mary", "NNP")],
    [("Jane", "NNP"), ("can", "MD"), ("pat", "VB"), ("Spot", "NNP")],
    [("Will", "NNP"), ("see", "VB"), ("Jane", "NNP")],
    [("Spot", "NNP"), ("can", "MD"), ("spot", "VB"), ("Mary", "NNP")],
    [("Mary", "NNP"), ("will", "MD"), ("see", "VB"), ("Will", "NNP")],
    [("Jane", "NNP"), ("will", "MD"), ("spot", "VB"), ("Spot", "NNP")],
    [("Will", "NNP"), ("can", "MD"), ("see", "VB"), ("Mary", "NNP")],
    [("Spot", "NNP"), ("will", "MD"), ("pat", "VB"), ("Jane", "NNP")],
    [("Mary", "NNP"), ("can", "MD"), ("spot", "VB"), ("will", "MD")],
    [("Jane", "NNP"), ("see", "VB"), ("Spot", "NNP")],
    [("Will", "NNP"), ("pat", "VB"), ("spot", "NNP")],
]

In [ ]:
def train_hmm(sentences):
    tag_count = Counter()
    trans_count = defaultdict(Counter)
    emit_count = defaultdict(Counter)
    vocab = set()
    tags = set()

    for sent in sentences:
        prev = "<s>"
        for word, tag in sent:
            tag_count[tag] += 1
            trans_count[prev][tag] += 1
            emit_count[tag][word] += 1
            vocab.add(word)
            tags.add(tag)
            prev = tag
        trans_count[prev]["</s>"] += 1

    # Start probabilities
    start_prob = {}
    total_starts = sum(trans_count["<s>"].values())
    for tag in tags:
        start_prob[tag] = trans_count["<s>"].get(tag, 0) / total_starts if total_starts > 0 else 1e-6

    # Transition probabilities (Laplace smoothing)
    trans_prob = defaultdict(dict)
    for prev in set(trans_count.keys()) | tags:
        total = sum(trans_count[prev].values()) + len(tags)
        for tag in tags:
            trans_prob[prev][tag] = (trans_count[prev].get(tag, 0) + 1) / total

    # Emission probabilities (Laplace smoothing)
    emit_prob = defaultdict(dict)
    for tag in tags:
        total = sum(emit_count[tag].values()) + len(vocab)
        for word in vocab:
            emit_prob[tag][word] = (emit_count[tag].get(word, 0) + 1) / total

    unk_emit = 1.0 / (len(vocab) * 10)

    return start_prob, trans_prob, emit_prob, list(tags), unk_emit, vocab


start_p, trans_p, emit_p, tag_list, unk_emit, vocab = train_hmm(train_sentences)

In [ ]:
def viterbi(words, start_p, trans_p, emit_p, tag_list, unk_emit):
    n = len(words)
    V = [{} for _ in range(n)]
    back = [{} for _ in range(n)]

    # Initialization
    for tag in tag_list:
        emiss = emit_p[tag].get(words[0], unk_emit)
        V[0][tag] = start_p.get(tag, 1e-8) * emiss
        back[0][tag] = None

    # Recursion
    for t in range(1, n):
        for tag in tag_list:
            max_prob = -math.inf
            max_prev = None
            emiss = emit_p[tag].get(words[t], unk_emit)

            for prev in tag_list:
                prob = V[t-1][prev] * trans_p[prev][tag] * emiss
                if prob > max_prob:
                    max_prob = prob
                    max_prev = prev

            V[t][tag] = max_prob
            back[t][tag] = max_prev

    # Termination
    best_path_prob = -math.inf
    best_last_tag = None
    for tag in tag_list:
        prob = V[n-1][tag] * trans_p[tag].get("</s>", 1e-8)
        if prob > best_path_prob:
            best_path_prob = prob
            best_last_tag = tag

    # Backtrack
    path = [None] * n
    path[n-1] = best_last_tag
    for t in range(n-2, -1, -1):
        path[t] = back[t+1][path[t+1]]

    return path

In [ ]:
test_sent = test_sentences[0]
words = [w for w, _ in test_sent]

predicted_tags = viterbi(words, start_p, trans_p, emit_p, tag_list, unk_emit)

print("Test sentence:")
print(" ".join(words))
print("\nPredicted tags:")
print(" ".join(predicted_tags))

print("\nFull prediction:")
for w, t in zip(words, predicted_tags):
    print(f"{w:10} {t}")

Test sentence:
Will can pot Mary

Predicted tags:
NNP MD VB NNP

Full prediction:
Will       NNP
can        MD
pot        VB
Mary       NNP


In [ ]:
def evaluate_hmm(test_sentences, start_p, trans_p, emit_p, tag_list, unk_emit):
    y_true = []
    y_pred = []

    print("Per-sentence evaluation:\n" + "="*65 + "\n")

    for i, sent in enumerate(test_sentences, 1):
        words = [w for w, _ in sent]
        gold_tags = [t for _, t in sent]

        predicted_tags = viterbi(words, start_p, trans_p, emit_p, tag_list, unk_emit)

        correct = all(g == p for g, p in zip(gold_tags, predicted_tags))

        print(f"Sentence {i}:")
        print("Words:     " + " ".join(words))
        print("Gold:      " + " ".join(gold_tags))
        print("Predicted: " + " ".join(predicted_tags))
        print("Correct:   " + ("Yes" if correct else "No"))
        print("-" * 65)

        y_true.extend(gold_tags)
        y_pred.extend(predicted_tags)

    # Global metrics
    print("\n" + "="*70)
    print("GLOBAL RESULTS")
    print("="*70)

    acc = accuracy_score(y_true, y_pred)
    total_tokens = len(y_true)
    correct_tokens = sum(1 for a, b in zip(y_true, y_pred) if a == b)

    print(f"Token accuracy: {acc:.4f}  ({correct_tokens}/{total_tokens} tokens correct)\n")

    print("Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    # Confusion matrix
    labels = sorted(set(tag_list))
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    print("\nConfusion Matrix (rows=gold, cols=predicted):")
    print("       " + "  ".join(f"{lbl:>6}" for lbl in labels))
    for i, row in enumerate(cm):
        print(f"{labels[i]:<6} " + "  ".join(f"{val:6}" for val in row))


evaluate_hmm(test_sentences, start_p, trans_p, emit_p, tag_list, unk_emit)

Per-sentence evaluation:

Sentence 1:
Words:     Will can pot Mary
Gold:      NNP MD VB NNP
Predicted: NNP MD VB NNP
Correct:   Yes
-----------------------------------------------------------------
Sentence 2:
Words:     Jane can pat Spot
Gold:      NNP MD VB NNP
Predicted: NNP MD VB NNP
Correct:   Yes
-----------------------------------------------------------------
Sentence 3:
Words:     Will see Jane
Gold:      NNP VB NNP
Predicted: NNP VB NNP
Correct:   Yes
-----------------------------------------------------------------
Sentence 4:
Words:     Spot can spot Mary
Gold:      NNP MD VB NNP
Predicted: NNP MD VB NNP
Correct:   Yes
-----------------------------------------------------------------
Sentence 5:
Words:     Mary will see Will
Gold:      NNP MD VB NNP
Predicted: NNP MD VB NNP
Correct:   Yes
-----------------------------------------------------------------
Sentence 6:
Words:     Jane will spot Spot
Gold:      NNP MD VB NNP
Predicted: NNP MD VB NNP
Correct:   Yes
--------------